# FarmAI vision: data → train → evaluate
Runtime → Change runtime type → **GPU** before you start. Run the cells in order.
Put the datasets you download by hand in Google Drive first (see `training/datasets.json`, field `get`):
`MyDrive/farmai_data/karaagro_maize/` and optionally `MyDrive/farmai_data/cassava_uganda/`.

In [ ]:
!git clone -q -b feature/vision-real-pipeline https://github.com/frost289/farmer-AI.git
%cd farmer-AI/ai-services/vision
# torch, torchvision, numpy and pillow come with Colab
!pip install -q onnx onnxruntime opencv-python-headless

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/farmai_data'
!mkdir -p data/raw
!cp -r {DRIVE}/karaagro_maize data/raw/ 2>/dev/null || echo "karaagro_maize not found in Drive"
!cp -r {DRIVE}/cassava_uganda data/raw/ 2>/dev/null || echo "cassava_uganda not found in Drive (optional)"

Tanzania maize from Zenodo: two zips of about 1.3 GB each (healthy + streak virus). Check the dataset's licence on its Zenodo page first.

In [ ]:
!python training/download_data.py --source tanzania_maize --extract

**Check the mapping before building.** Every folder should map to the label you expect; anything wrong means editing `training/datasets.json`.

In [ ]:
!python training/prepare_data.py --dry-run

In [ ]:
!python training/prepare_data.py --force

Read the `warnings` above: labels with no images cannot be learned, and tiny classes give weak, noisy results.

In [ ]:
!python training/train_classifier.py --data data --out models --version v0.2.0 --epochs 15

Evaluate on the **test** folder, which training never saw. Look at per-class recall, the low_confidence rate, and accuracy when confident.

In [ ]:
!python training/evaluate.py --data data/test

In [ ]:
!mkdir -p {DRIVE}/models && cp models/classifier.onnx models/labels.json {DRIVE}/models/
print('Saved to Drive. Copy both files into ai-services/vision/models/ and restart the service.')